# Fase 3 – Esportazione per la ricerca nel browser

Calcola gli embeddings dei chunk con **multilingual-e5-small** e prepara in `/kaggle/working/web_assets`
i file per `web/`: chunk (con la pagina corrispondente nel PDF italiano), embeddings in float16,
configurazione (stopword, regex, parametri identici a Python), set di valutazione e riferimenti per il test di parita'.

**Impostazioni:** GPU T4, Internet ON. **Input:** Dataset del parsing inglese e Dataset del parsing italiano.

In [ ]:
!pip install -q "qrt-rag[eval] @ git+https://github.com/mat126/rag-qrt-SolvencyII.git"

In [ ]:
from pathlib import Path

import torch
from sentence_transformers import SentenceTransformer

from qrt_rag.alignment import align_pages, alignment_report
from qrt_rag.chunking import build_chunks
from qrt_rag.evaluation import load_eval_set
from qrt_rag.index import PrefixedEncoder, embed_chunks
from qrt_rag.parsing import load_pages
from qrt_rag.web_export import export_web

LANG = "en"
WORK = Path("/kaggle/working")
PARSED_INPUT = Path("/kaggle/input/datasets/mat126/parsing-rag-qrt-eng")     # versione inglese
PARSED_REF = Path("/kaggle/input/datasets/mat126/parsing-rag-qrt-ivass")     # versione italiana
OUT = WORK / "web_assets"

# Nomi con cui caricherai i due PDF in web/assets/
PDFS = {"en": "regulation_en.pdf", "it": "regulation_it.pdf"}

# Modello scelto nella fase 2; "web" e' la versione ONNX per transformers.js
MODEL = {
    "hf": "intfloat/multilingual-e5-small",
    "web": "Xenova/multilingual-e5-small",
    "q_prefix": "query: ",
    "p_prefix": "passage: ",
    "pooling": "mean",
}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DEVICE)

In [ ]:
pages = load_pages(PARSED_INPUT)
chunks = build_chunks(pages, lang=LANG)
cases = load_eval_set(lang=LANG)
print(f"Chunk: {len(chunks)} | casi: {len(cases)}")

In [ ]:
# Pagine corrispondenti nel PDF italiano: per le istruzioni tramite la chiave
# allegato + template + codice cella, per il resto con articoli, template o scarto vicino
chunks_it = build_chunks(load_pages(PARSED_REF), lang="it")
ref_pages = align_pages(chunks, chunks_it)
_ = alignment_report(chunks, ref_pages)

In [ ]:
model = SentenceTransformer(MODEL["hf"], device=DEVICE)
print("Lunghezza massima:", model.max_seq_length, "token")
E = embed_chunks(model, chunks, batch_size=64, prefix=MODEL["p_prefix"])

In [ ]:
# Esportazione: stampa le dimensioni dei file e le metriche Python con embeddings float16,
# che devono coincidere con quelle della fase 2 (hit@5 0,968, MRR 0,620)
config = export_web(OUT, chunks, E, PrefixedEncoder(model, MODEL["q_prefix"]), MODEL, cases,
                    lang=LANG, pdfs=PDFS, ref_lang="it", ref_pages=ref_pages)

In [ ]:
# Archivio da scaricare dal pannello Output e da estrarre in web/assets/ del repository
import shutil
shutil.make_archive(str(WORK / "web_assets"), "zip", OUT)
print("Creato", WORK / "web_assets.zip")